$$\textit{To learn is virtue, to seek is divine}$$

In [3]:
# An equivalent way to implement this, is to set abs_
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper

from sorl.trainer_ablate import SoRLTrainerv2, SoRLTrainerv3
from sorl.trainer_ablate import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
import torch
import torch.nn.functional as F
from sorl.selfroute import _find_similar_magnitude_dims

nl_vocab  = int(model.vocab_sizes[0].item())
abs_vocab = int(model.vocab_sizes[1].item())   # includes placeholder at offset 0
n_abs_real = abs_vocab - 1                     # real abstract tokens (skip placeholder)

# (1). Initialize abstract lm_head rows: "similar magnitude" one-hot projection
#      Each abstract token k routes to exactly one hidden dim, chosen so that
#      all selected dims have uniformly-sized column norms in the NL lm_head.
with torch.no_grad():
    lm_w = model.model.lm_head.weight                      # (total_vocab, hidden)
    selected_dims, importances, cv = _find_similar_magnitude_dims(
        lm_w[:nl_vocab].float(), n_abs_real)
    print(f"Selected {n_abs_real} dims | "
          f"importance=[{importances.min():.4f}, {importances.max():.4f}] | CV={cv:.6f}")

    lm_w.data[nl_vocab:] = 0.0                             # zero all abstract rows
    for k in range(1, abs_vocab):                          # skip placeholder (k=0)
        lm_w.data[nl_vocab + k, selected_dims[k - 1].item()] = 1.0

print(f"lm_head rows [{nl_vocab}:{nl_vocab + abs_vocab}] initialized (similar-magnitude one-hot)")

# (2). Gradient hook: abstract rows are trainable; NL rows stay frozen.
#      We register a hook on lm_head.weight that zeros gradients for [:nl_vocab].
def _freeze_nl_rows(grad):
    g = grad.clone()
    g[:nl_vocab] = 0.0
    return g

_lm_head_hook = lm_w.register_hook(_freeze_nl_rows)
print(f"Hook registered: lm_head NL rows [:{nl_vocab}] are frozen; abstract rows [{nl_vocab}:] trainable")

Selected 128 dims | importance=[10.8605, 10.9294] | CV=0.001870
lm_head rows [151936:152065] initialized (similar-magnitude one-hot)


In [ ]:
# REINFORCE-style SoRL
# Policy  π(a | ctx): abstract token selection during recursion
# Reward  R_i = -traj_loss_i  (higher = NL tokens predicted better given abstractions)
# Baseline: group mean per sample  (REINFORCE with leave-one-out)
# Loss    = -E[ A_i · log π(a_i | ctx) ]   where A_i = (R_i - mean) / std

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sorl.sorl_trainer import (
    infer_insert_mask, expand_prompt_len, insert_tokens_with_padding,
)
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# ── Config ──────────────────────────────────────────────────────────────────
N  = 4          # rollouts per sample
K  = 4          # abstract tokens inserted
MAX_ITERS   = 2
TEMPERATURE = 1.0
MEM_SPAN    = 1792
LR          = 1e-5
BATCH_SIZE  = 2
LOG_EVERY   = 10
MAX_STEPS   = 300

dataset_name = "gsm8k"
max_length   = 256
# ────────────────────────────────────────────────────────────────────────────

train_ds = get_dataset(dataset_name, split="train", tokenizer=tokenizer, max_length=max_length)
val_ds   = get_dataset(dataset_name, split="test",  tokenizer=tokenizer, max_length=max_length)
dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

pad_id     = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history   = {"step": [], "loss": [], "reward_mean": [], "reward_std": [], "n_abs": []}

model.train()
global_step = 0
print(f"REINFORCE SoRL | K={K} | N={N} | temp={TEMPERATURE} | max_steps={MAX_STEPS}")
print("=" * 60)

for batch in dl:
    ids  = batch["input_ids"].to(device)
    attn = batch["attention_mask"].to(device)
    pl   = batch["prompt_len"].to(device)
    B    = ids.shape[0]

    # ── 1. Insert abstract token slots & run N rollouts (no grad) ───────────
    ins_mask = infer_insert_mask(ids, K, attn)
    exp_pl   = expand_prompt_len(pl, ins_mask)
    exp_data, exp_mask = insert_tokens_with_padding(
        ids, attn, ins_mask, model.vocab_sizes[0], pad_id)

    rep_data = exp_data.repeat_interleave(N, dim=0)    # (B*N, L)
    rep_mask = exp_mask.repeat_interleave(N, dim=0)
    rep_pl   = exp_pl.repeat_interleave(N, dim=0)

    with torch.no_grad():
        all_data, ppt, _ = model.recursion(
            rep_data, rep_mask,
            max_iterations=MAX_ITERS,
            memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN,
            temperature=TEMPERATURE, prompt_len=rep_pl,
        )
        # Reward = -mean NL traj loss per rollout  (shape: B*N)
        valid  = (ppt != 0).float()
        reward = -(ppt.sum(1) / valid.sum(1).clamp(min=1))

    # ── 2. Group-normalised advantage (leave-one-out baseline) ──────────────
    r_g   = reward.view(B, N)
    mean_r = r_g.mean(1, keepdim=True)
    std_r  = r_g.std(1, keepdim=True).clamp(min=1e-6)
    adv    = ((r_g - mean_r) / std_r).view(-1)         # (B*N,)

    # ── 3. log π(abstract_choices | context)  under current policy ──────────
    outputs = model(
        input_ids=all_data, attention_mask=rep_mask,
        memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN,
    )
    shift_logits = outputs.logits[:, :-1, :].contiguous()   # (B*N, L-1, V)
    shift_ids    = all_data[:, 1:].contiguous()              # (B*N, L-1)
    shift_attn   = rep_mask[:, 1:].float()

    # Positions where an abstract token was placed
    abs_pos = (shift_ids >= base_vocab).float() * shift_attn  # (B*N, L-1)

    # Restrict softmax to abstract vocab; gather log prob of the chosen token
    abs_logits = shift_logits.clone()
    abs_logits[..., :base_vocab] = -float("inf")
    log_probs  = F.log_softmax(abs_logits, dim=-1)

    safe_ids   = shift_ids.clone()
    safe_ids[shift_ids < base_vocab] = base_vocab              # safe gather index
    per_tok_lp = log_probs.gather(2, safe_ids.unsqueeze(-1)).squeeze(-1)  # (B*N, L-1)
    per_tok_lp = per_tok_lp * abs_pos

    n_abs_toks = abs_pos.sum(1).clamp(min=1)
    log_pi     = per_tok_lp.sum(1) / n_abs_toks               # (B*N,)

    # ── 4. REINFORCE update ─────────────────────────────────────────────────
    loss = -(adv.detach() * log_pi).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    global_step += 1

    if global_step % LOG_EVERY == 0:
        r_mean = reward.mean().item()
        r_std  = reward.std().item()
        n_abs_avg = abs_pos.sum(1).mean().item()
        print(f"step {global_step:4d} | loss={loss.item():.4f} "
              f"| reward μ={r_mean:.3f} σ={r_std:.3f} | abs_toks/seq={n_abs_avg:.1f}")
        history["step"].append(global_step)
        history["loss"].append(loss.item())
        history["reward_mean"].append(r_mean)
        history["reward_std"].append(r_std)
        history["n_abs"].append(n_abs_avg)

    if global_step >= MAX_STEPS:
        break

print(f"\nDone. {global_step} steps.")

REINFORCE SoRL | K=4 | N=4 | temp=1.0 | max_steps=300

Done. 0 steps.


In [ ]:
# GRPO kernel
# ─────────────────────────────────────────────────────────────────────────────
# Reward  : R_i = -traj_loss_i  = -(ppt.sum(1) / valid.sum(1).clamp(min=1))
# Loss    : -E[ min(r_t · A,  clip(r_t, 1-ε, 1+ε) · A) ]        [p(a|s)]
#           r_t = π_new(a|ctx) / π_old(a|ctx)
# Train   : full model; NL lm_head rows frozen via hook from cell 3
#
# Outer loop : new rollouts → reference old_log_pi → fixed advantage
# Inner loop : N_INNER gradient steps on the SAME rollout data
#              Skipped if adv_std < ADV_STD_MIN (all rollouts same quality)
#
# To start from a v7 checkpoint:
#   model = SorlModelWrapper.from_pretrained("<ckpt_dir>", abstract_vocab_size_list=[128])
#   model = model.to(device)
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sorl.sorl_trainer import infer_insert_mask, expand_prompt_len, insert_tokens_with_padding
from data.pt_dataset import get_dataset, collate_fn

# ── Config ───────────────────────────────────────────────────────────────────
N           = 4
K           = 4
N_INNER     = 5
MAX_ITERS   = 2
TEMPERATURE = 1.0
MEM_SPAN    = 1792
LR          = 1e-5
BATCH_SIZE  = 2
LOG_EVERY   = 10
MAX_STEPS   = 300
CLIP_EPS    = 0.2
ENT_COEF    = 0.01
ADV_STD_MIN = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

# ── Sanity-check: abort early if model weights are already NaN ────────────────
_nan_params = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
if _nan_params:
    raise RuntimeError(
        f"Model weights contain NaN — reload model (re-run cells 2+3).\n"
        f"Affected: {_nan_params[:5]}")
print("Weight NaN check: OK")

train_ds  = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
dl_grpo   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

pad_id     = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())
nl_vocab   = base_vocab

# Ensure NL lm_head rows are frozen
_grpo_hook = None
def _freeze_nl_grpo(grad):
    g = grad.clone(); g[:nl_vocab] = 0.0; return g
if not any(True for _ in model.model.lm_head.weight._backward_hooks.values()):
    _grpo_hook = model.model.lm_head.weight.register_hook(_freeze_nl_grpo)
    print("Hook registered: NL lm_head rows frozen")
else:
    print("Hook already active from cell 3")

optimizer_grpo = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
hist_grpo = {"step": [], "loss": [], "loss_pg": [], "loss_ent": [],
             "reward": [], "adv_std": [], "ratio_mean": [], "clip_frac": []}

# ── Helpers ──────────────────────────────────────────────────────────────────
def _abs_log_pi(logits, ids, attn):
    sl = logits[:, :-1, :].contiguous()
    si = ids[:, 1:].contiguous()
    sa = attn[:, 1:].float()
    pos = (si >= base_vocab).float() * sa
    lg  = sl.clone(); lg[..., :base_vocab] = -float("inf")
    sid = si.clone(); sid[si < base_vocab] = base_vocab
    tok_lp = F.log_softmax(lg, dim=-1).gather(2, sid.unsqueeze(-1)).squeeze(-1) * pos
    return tok_lp.sum(1) / pos.sum(1).clamp(min=1), pos.sum(1)

def _abs_entropy(logits, attn):
    sl = logits[:, :-1, :].contiguous()
    sa = attn[:, 1:].float()
    lg = sl.clone(); lg[..., :base_vocab] = -float("inf")
    p  = F.softmax(lg, dim=-1); lp = F.log_softmax(lg, dim=-1)
    ent = -(p * lp).sum(-1) * sa
    return ent.sum(1) / sa.sum(1).clamp(min=1)

_skipped = 0; _inner_skipped = 0
model.train()
global_step = 0
print(f"GRPO | K={K} N={N} N_INNER={N_INNER} ε={CLIP_EPS} "
      f"ent={ENT_COEF} adv_min={ADV_STD_MIN} max_steps={MAX_STEPS}")
print("=" * 70)

for batch in dl_grpo:
    ids  = batch["input_ids"].to(device)
    attn = batch["attention_mask"].to(device)
    pl   = batch["prompt_len"].to(device)
    B    = ids.shape[0]

    # ── (I) Outer: rollout + reference policy (no grad) ─────────────────────
    ins_mask = infer_insert_mask(ids, K, attn)
    exp_pl   = expand_prompt_len(pl, ins_mask)
    exp_data, exp_mask = insert_tokens_with_padding(
        ids, attn, ins_mask, model.vocab_sizes[0], pad_id)

    rep_data = exp_data.repeat_interleave(N, dim=0)
    rep_mask = exp_mask.repeat_interleave(N, dim=0)
    rep_pl   = exp_pl.repeat_interleave(N, dim=0)

    with torch.no_grad():
        all_data, ppt, _ = model.recursion(
            rep_data, rep_mask,
            max_iterations=MAX_ITERS,
            memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN,
            temperature=TEMPERATURE, prompt_len=rep_pl,
        )
        valid  = (ppt != 0).float()
        reward = -(ppt.sum(1) / valid.sum(1).clamp(min=1))          # (B*N,)

        out_ref = model(input_ids=all_data, attention_mask=rep_mask,
                        memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN)
        old_log_pi, _ = _abs_log_pi(out_ref.logits, all_data, rep_mask)
        del out_ref

    # ── NaN guard ─────────────────────────────────────────────────────────────
    if torch.isnan(reward).any() or torch.isinf(reward).any():
        _skipped += 1
        if _skipped <= 5:
            print(f"[skip] step {global_step} — NaN/inf in reward")
        continue

    # Advantage: group-normalised, frozen across inner steps
    r_g    = reward.view(B, N)
    mean_r = r_g.mean(1, keepdim=True)
    std_r  = r_g.std(1, keepdim=True)
    adv_std_mean = std_r.mean().item()
    adv    = ((r_g - mean_r) / std_r.clamp(min=1e-6)).view(-1).detach()

    # ── Skip inner loop if reward variance is too low ─────────────────────────
    if adv_std_mean < ADV_STD_MIN:
        _inner_skipped += 1
        global_step += 1
        continue

    # ── (II) Inner: N_INNER gradient steps on same rollout data ─────────────
    for inner in range(N_INNER):
        out_new    = model(input_ids=all_data, attention_mask=rep_mask,
                           memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN)
        new_log_pi, _ = _abs_log_pi(out_new.logits, all_data, rep_mask)
        entropy        = _abs_entropy(out_new.logits, rep_mask)

        log_ratio = (new_log_pi - old_log_pi.detach()).clamp(-4.0, 4.0)
        ratio     = log_ratio.exp()

        pg_loss1 = ratio * adv
        pg_loss2 = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * adv
        loss_pg  = -torch.min(pg_loss1, pg_loss2).mean()
        loss_ent = -ENT_COEF * entropy.mean()
        loss     = loss_pg + loss_ent

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"[skip inner] step {global_step} inner {inner} — NaN loss")
            break

        optimizer_grpo.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_grpo.step()

    global_step += 1

    if global_step % LOG_EVERY == 0:
        clip_frac = ((ratio - 1).abs() > CLIP_EPS).float().mean().item()
        print(f"step {global_step:4d} | loss={loss.item():.4f} "
              f"(pg={loss_pg.item():.4f} ent={loss_ent.item():.4f}) "
              f"| reward μ={reward.mean():.3f} σ={reward.std():.3f} "
              f"| adv_std={adv_std_mean:.4f} | ratio={ratio.mean():.3f} "
              f"| clip={clip_frac:.2f} | skip_r={_skipped} skip_i={_inner_skipped}")
        hist_grpo["step"].append(global_step)
        hist_grpo["loss"].append(loss.item())
        hist_grpo["loss_pg"].append(loss_pg.item())
        hist_grpo["loss_ent"].append(loss_ent.item())
        hist_grpo["reward"].append(reward.mean().item())
        hist_grpo["adv_std"].append(adv_std_mean)
        hist_grpo["ratio_mean"].append(ratio.mean().item())
        hist_grpo["clip_frac"].append(clip_frac)

    if global_step >= MAX_STEPS:
        break

print(f"\nDone. {global_step} steps | skip_reward={_skipped} skip_inner={_inner_skipped}.")
if _grpo_hook is not None:
    _grpo_hook.remove()